# Fresh-world Monte Carlo comparison
Each repetition generates a new synthetic world. Algorithms share that world and observe only their own selected rewards. Run top to bottom; edit chart cells without retraining.

In [ ]:
import os, sys, json, copy
from pathlib import Path
for key in ('OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','VECLIB_MAXIMUM_THREADS'):
    os.environ[key]='1'
os.environ.setdefault('MPLCONFIGDIR',str(Path.cwd()/'.cache/matplotlib'))
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents]
          if (p/'contextual_bandits/simulator.py').exists())
sys.path.insert(0,str(ROOT))
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink
from contextual_bandits.simulator import SCENARIOS, FEATURE_NAMES, from_records
from contextual_bandits.interface import ContextPolicy
from contextual_bandits.workbench import (
    default_settings, make_world, dataset_path, world_dates, event_table, run_experiment,
    summary_table, paired_table, save_experiment, load_experiment,
    plot_world, plot_rewards, plot_regret, plot_choices, plot_shift_response)
plt.rcParams.update({'figure.dpi':110,'font.size':10,'axes.spines.top':False,'axes.spines.right':False})
defaults=default_settings()
print('Python:',sys.version.split()[0])
print('Configuration loaded. All runs save their own data and settings.')
print('Scenarios:',', '.join(SCENARIOS))

In [ ]:
MC_REPEATS=4
MC_SCENARIOS=['stationary','recurring','birth_contextual']
METHODS=['fixed','ucb','sw_ucb','discount_ucb','ts','epsilon','softmax','dr_softmax']
HORIZON=18250
SEED=20261106
CONFIGS=default_settings()['configs']
FIXED_ARMS=default_settings()['fixed']


## Run independent world blocks
This small comparison uses starting configurations. Use the staged runner for independent tuning, validation, a precision pilot and frozen confirmation.

In [ ]:
mc=run_experiment(MC_SCENARIOS,METHODS,CONFIGS,FIXED_ARMS,repeats=MC_REPEATS,horizon=HORIZON,namespace=SEED)
display(summary_table(mc).round(5))
display(pd.DataFrame(mc.metadata['datasets'])[['world','scenario','path']])
print('Elapsed seconds:',round(mc.metadata['seconds'],2))

## Reward, regret and regime response
Shading is a pointwise interval across independent worlds, not a prediction interval or simultaneous band.

In [ ]:
SCENARIO_TO_PLOT='recurring'
SMOOTHING=250
plot_rewards(mc,SCENARIO_TO_PLOT,window=SMOOTHING);plt.show()
plot_regret(mc,SCENARIO_TO_PLOT,window=SMOOTHING);plt.show()
if mc.worlds[0,SCENARIO_TO_PLOT].shifts:
    plot_shift_response(mc,SCENARIO_TO_PLOT,before=500,after=1000,window=100);plt.show()
display(paired_table(mc,'ucb').round(5))

## Save and reload

In [ ]:
directory=save_experiment(mc)
restored=load_experiment(directory)
pd.testing.assert_frame_equal(summary_table(mc),summary_table(restored))
print('Saved and verified:',directory)
